# A professional prediction workflow with `Pipeline`

**Thursday 1 October · 12:30–15:00 · work in pairs**

**Question.** After qualifying, which drivers might finish in the top ten? This is an **invented teaching dataset**, not Formula 1 results. It contains 18 race weekends and 20 fictional drivers per weekend (360 rows). One row represents one driver in one weekend. There are missing numbers, a missing category, and a new circuit category later in time.

**Why this notebook?** In a manual workflow you must remember to fit each imputer, encoder and scaler on the right rows, then apply the same fitted objects in exactly the same order later. A `Pipeline` packages those steps with the model. `fit(X_train, y_train)` learns them together; `predict(X_later)` reuses them. It also lets a parameter search refit the **whole procedure** inside each temporal fold. A pipeline does **not** decide which rows are training or whether a feature is available at prediction time.

**By 15:00, hand in:** one supplied Thursday worksheet per pair and one individual exit ticket. Use short labels in English or Spanish. Predict before running each marked cell. If execution fails, read the saved outputs and explain them.

**Route:** 1 inspect and split → 2 build → 3 fit and inspect → 4 validate → 5 compare temporal folds → 6 apply to new rows → 7 make one controlled change and diagnose limits. Work through the stop points on your worksheet.

## 1. Inspect the information and choose the split

Available **after qualifying, before the race**: qualifying position, practice time gap, previous top-ten rate, grid penalty, circuit type and rain forecast. The target `top10` is the race outcome; it is **never an input feature**. `driver_id` and `week` identify rows and control the split; they are not model features.

**Stop 1.** Predict how many rows belong to training (weeks 1–10), validation (11–14), and the reserved test (15–18). Why must we split before fitting missing-value rules?

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV

path = Path("w04_race_weekends_synthetic_v3.csv")
if not path.exists():
    raise FileNotFoundError("Open the notebook in the folder containing its CSV file.")
data = pd.read_csv(path)
print("Rows, columns:", data.shape)
print(data.head(5).to_string(index=False))
print("The displayed rows are from week 1; later outcomes are not inspected here.")

Rows, columns: (360, 9)
 week driver_id  qualifying_position  practice_gap_s  previous_top10_rate  grid_penalty circuit_type rain_forecast  top10
    1       D01                    4           0.302                0.355             0       street            no      1
    1       D02                    3           0.168                0.448             0       street            no      1
    1       D03                   13           0.981                0.676             0       street            no      0
    1       D04                    6           0.381                0.553             0       street            no      1
    1       D05                    1           0.020                0.630             5       street            no      1
The displayed rows are from week 1; later outcomes are not inspected here.


In [2]:
train = data.loc[data.week <= 10].copy()
valid = data.loc[data.week.between(11, 14)].copy()
test_reserved = data.loc[data.week.between(15, 18)].copy()  # Keep sealed today.
numeric = ["qualifying_position", "practice_gap_s", "previous_top10_rate", "grid_penalty"]
categorical = ["circuit_type", "rain_forecast"]
features = numeric + categorical
X_train, y_train = train[features], train.top10
X_valid, y_valid = valid[features], valid.top10
print("Training / validation / reserved test:", len(train), len(valid), len(test_reserved))
print("Validation circuits:", sorted(valid.circuit_type.unique()))
print("Training circuits:", sorted(train.circuit_type.unique()))
print("Missing training inputs:", train[features].isna().sum().to_dict())
print("Top-ten outcomes in each training weekend:", train.groupby("week").top10.sum().unique().tolist())
assert train.week.max() < valid.week.min() < test_reserved.week.min()
assert "top10" not in features and "week" not in features

Training / validation / reserved test: 200 80 80
Validation circuits: ['high_speed', 'road', 'street', 'technical']
Training circuits: ['high_speed', 'road', 'street']
Missing training inputs: {'qualifying_position': 0, 'practice_gap_s': 17, 'previous_top10_rate': 13, 'grid_penalty': 0, 'circuit_type': 0, 'rain_forecast': 14}
Top-ten outcomes in each training weekend: [10]


## 2. Build one estimator from several steps

The **numeric branch** replaces missing values with the **training median** and standardises each feature using training statistics. The **categorical branch** replaces a missing category with the most frequent training category and one-hot encodes it. `handle_unknown="ignore"` allows a later `technical` circuit even though none appears in training. Both branches feed a logistic regression classifier.

`ColumnTransformer` runs different preprocessing on different columns; the outer `Pipeline` joins preprocessing and model. The names `prep`, `num`, `cat` and `model` let us inspect and tune steps. `__` reaches into a named step, such as `model__C`.

**Stop 2.** Which operation actually learns medians, scales, categories and model coefficients: constructing the Python objects or calling `fit`?

In [3]:
numeric_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical),
])
model = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
print(model)
print("Example tunable parameter:", model.get_params()["model__C"])
print("No fitting has happened yet.")

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['qualifying_position',
                                                   'practice_gap_s',
                                                   'previous_top10_rate',
                                                   'grid_penalty']),
                                                 ('cat',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                  

## 3. Fit on training only, then inspect what was learned

This is the only `fit` call for the first candidate. The fitted preprocessing lives inside `model.named_steps["prep"]`. In particular, `statistics_` gives the four learned numeric medians in the order of `numeric`.

**Stop 3.** Predict whether a validation row can change those stored medians when we call `predict`.

In [4]:
model.fit(X_train, y_train)
learned = model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_.copy()
print("Training medians:", dict(zip(numeric, learned.round(3))))
print("Transformed feature count:", len(model.named_steps["prep"].get_feature_names_out()))
print("First eight transformed names:", model.named_steps["prep"].get_feature_names_out()[:8].tolist())
assert len(learned) == len(numeric)
assert not any(pd.isna(learned))

Training medians: {'qualifying_position': np.float64(10.5), 'practice_gap_s': np.float64(0.695), 'previous_top10_rate': np.float64(0.493), 'grid_penalty': np.float64(0.0)}
Transformed feature count: 9
First eight transformed names: ['num__qualifying_position', 'num__practice_gap_s', 'num__previous_top10_rate', 'num__grid_penalty', 'cat__circuit_type_high_speed', 'cat__circuit_type_road', 'cat__circuit_type_street', 'cat__rain_forecast_no']


## 4. Validate the fitted procedure

A **baseline** is a fair simple comparison: it always predicts the most frequent class learned from training. We report accuracy and balanced accuracy because class proportions can make accuracy misleading. The validation period helps us compare approaches. The reserved test stays unopened.

`model.predict(X_valid)` performs the fitted preprocessing and classifier prediction in one call. We do not manually impute or encode validation.

**Stop 4.** Before running, identify the line that would be unsafe: `preprocess.fit_transform(data[features])` before the time split, or `model.predict(X_valid)` after fitting on training? Explain in one sentence.

In [5]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_valid)
valid_pred = model.predict(X_valid)
print("Baseline accuracy / balanced accuracy:", round(accuracy_score(y_valid, baseline_pred),3), round(balanced_accuracy_score(y_valid, baseline_pred),3))
print("Pipeline accuracy / balanced accuracy:", round(accuracy_score(y_valid, valid_pred),3), round(balanced_accuracy_score(y_valid, valid_pred),3))
print("Stored medians unchanged after predict:", bool((learned == model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))
assert (learned == model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()

Baseline accuracy / balanced accuracy: 0.5 0.5
Pipeline accuracy / balanced accuracy: 0.863 0.863
Stored medians unchanged after predict: True


## 5. The professional advantage: tune the whole procedure in temporal folds

Within the **training period only**, compare `model__C` values 0.1, 1 and 10. Two expanding folds are explicit: weeks 1–6 → 7–8, then weeks 1–8 → 9–10. In each fold, `GridSearchCV` clones the **entire pipeline** and fits its imputer, encoder, scaler and model again on that fold's earlier rows. A manual workflow can easily fit preprocessing once on all weeks 1–10 before cross-validation, leaking later-fold information.

The winning pipeline is refitted on weeks 1–10. We may then inspect weeks 11–14 once for this teaching comparison. We still do not open the reserved test.

**Stop 5.** In fold 1, which weeks may teach the imputer its medians? Where does `model__C` point?

In [6]:
folds = []
for last_train_week, first_val_week, last_val_week in [(6,7,8),(8,9,10)]:
    tr_idx = train.index[train.week <= last_train_week].to_numpy()
    va_idx = train.index[train.week.between(first_val_week,last_val_week)].to_numpy()
    folds.append((tr_idx, va_idx))
    print(f"Fold: train weeks 1–{last_train_week} ({len(tr_idx)} rows), validate {first_val_week}–{last_val_week} ({len(va_idx)} rows)")
search = GridSearchCV(model, {"model__C": [0.1, 1.0, 10.0]}, cv=folds, scoring="balanced_accuracy", refit=True)
search.fit(X_train, y_train)
print("Mean fold balanced accuracy by C:", [(p["model__C"],round(s,3)) for p,s in zip(search.cv_results_["params"],search.cv_results_["mean_test_score"])])
print("Selected C:", search.best_params_["model__C"])
print("Refitted pipeline validation balanced accuracy:", round(balanced_accuracy_score(y_valid, search.predict(X_valid)),3))
assert all(max(train.iloc[tr].week) < min(train.iloc[va].week) for tr,va in folds)

Fold: train weeks 1–6 (120 rows), validate 7–8 (40 rows)
Fold: train weeks 1–8 (160 rows), validate 9–10 (40 rows)
Mean fold balanced accuracy by C: [(0.1, np.float64(0.912)), (1.0, np.float64(0.888)), (10.0, np.float64(0.85))]
Selected C: 0.1
Refitted pipeline validation balanced accuracy: 0.875


## 6. Apply one fitted object to later rows, including an unseen category

`technical` appears later and was absent from training. With `handle_unknown="ignore"`, the encoder keeps the learned training columns and gives zeros for that category's one-hot columns; it does **not** learn a new category during prediction. A missing numeric input receives a stored training median. This is the same operation we would call when new rows arrive.

**Stop 6.** Change only the first row's `practice_gap_s` to missing in the copy below. Will the model's stored median change? Are individual predictions guaranteed to remain the same?

In [7]:
later = X_valid.iloc[[0,1,2]].copy()
later.loc[later.index[0], "circuit_type"] = "technical"
later.loc[later.index[0], "practice_gap_s"] = float("nan")
before = search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_.copy()
print("New-row features:", later.to_string(index=False))
print("Predicted top-ten labels:", search.predict(later).tolist())
print("Stored medians still the same:", bool((before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))
assert (before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()

New-row features:  qualifying_position  practice_gap_s  previous_top10_rate  grid_penalty circuit_type rain_forecast
                   6             NaN                0.264             0    technical            no
                  12           0.887                  NaN             0    technical            no
                  16           0.939                0.631             0    technical            no
Predicted top-ten labels: [1, 0, 0]
Stored medians still the same: True


## 7. Your change: edit one input, keep the fitted pipeline

The next cell starts with `new_qualifying_position = 20`. Before running, predict whether the top-ten label for the first later row will change. **Your pair's edit:** after the first run, set this number to **1**, rerun only this cell, and record the second outcome. Do **not** rerun `fit` or the parameter search. Explain why stored preprocessing values remain fixed even if a prediction changes.

In [8]:
new_qualifying_position = 20  # YOUR EDIT: after the first run, change 20 to 1 and rerun this cell.
scenario = later.iloc[[0]].copy()
scenario.loc[scenario.index[0], "qualifying_position"] = new_qualifying_position
print("Original qualifying position:", int(later.iloc[0].qualifying_position))
print("Scenario qualifying position:", new_qualifying_position)
print("Original / scenario predictions:", int(search.predict(later.iloc[[0]])[0]), int(search.predict(scenario)[0]))
print("Stored medians still the same:", bool((before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))

Original qualifying position: 6
Scenario qualifying position: 20
Original / scenario predictions: 1 0
Stored medians still the same: True


## 8. Diagnose the limits, then hand in

The one-line operation `search.predict(new_rows)` is useful because its preprocessing and model stay together. It does **not** select a valid time split, detect that a feature was measured after the race, or establish performance for every future circuit. `handle_unknown="ignore"` avoids an error for a new category; it does not prove good predictions on that category. The reserved test weeks 15–18 were not evaluated in this class.

**Pair discussion:** Which two checks must still be done by people? Give one possible leakage source outside the pipeline. Point to the object that contains the fitted median. Complete the worksheet and each person's exit ticket.

Reference: [scikit-learn Pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

## 9. Lab 2 · Can one pre-race feature improve the top-10 prediction?

**Pair:** Andy Villarroel (proposal, feature, reproducibility) · Agustín Reyes Pereira (comparison, decision, AI audit).

**Split (unchanged):** train weeks 1–10 · validation weeks 11–14 · reserved test weeks 15–18 (**not scored in this lab**).

### Proposal 
**Feature:** `estimated_grid_position`. **Formula:** `qualifying_position + grid_penalty`, computed row by row with no fitting. **Source columns:** `qualifying_position` is known after Saturday qualifying; `grid_penalty` is announced by the stewards before the race starts. Both exist before the race, so the feature is available at prediction time. It never uses `top10`, finishing position or any race result. **Idea:** the actual starting slot may predict a top-10 finish better than qualifying alone. **Expected direction:** little or no change in validation balanced accuracy (0.00 to +0.02), because logistic regression already receives both source columns and can learn their sum; the new column is an exact linear combination of them.

*Word count: 104.*

## 10. Create exactly one candidate feature

The feature is a row-wise sum, so it learns nothing from other rows and cannot leak across the split. We still add it to each split separately (on copies) so the reserved test is never touched. The assertions check that only pre-race columns are used.

**Stop 9.** Predict: will any training row get a value above 20?

In [9]:
CANDIDATE = "estimated_grid_position"
SOURCE_COLS = ["qualifying_position", "grid_penalty"]  # both known before the race
FORBIDDEN = {"top10", "week", "driver_id"}  # target and identifiers never feed the feature

def add_estimated_grid(df):
    return df.assign(**{CANDIDATE: df["qualifying_position"] + df["grid_penalty"]})

assert not FORBIDDEN & set(SOURCE_COLS)
assert all(col in features for col in SOURCE_COLS), "Sources must already be accepted pre-race inputs."

train_f = add_estimated_grid(train)
valid_f = add_estimated_grid(valid)
# test_reserved is deliberately NOT transformed or scored.

numeric_with = numeric + [CANDIDATE]
features_without = numeric + categorical
features_with = numeric_with + categorical
assert set(features_with) - set(features_without) == {CANDIDATE}, "Exactly one new feature."

print("Missing values in new feature (train / valid):", train_f[CANDIDATE].isna().sum(), valid_f[CANDIDATE].isna().sum())
print("Training range:", train_f[CANDIDATE].min(), "to", train_f[CANDIDATE].max())
print("Training rows with a grid penalty:", int((train_f.grid_penalty > 0).sum()), "| validation:", int((valid_f.grid_penalty > 0).sum()))
print("Correlation with qualifying_position (train):", round(train_f[CANDIDATE].corr(train_f.qualifying_position), 3))
print(train_f.loc[train_f.grid_penalty > 0, ["week", "driver_id", "qualifying_position", "grid_penalty", CANDIDATE]].head(5).to_string(index=False))

Missing values in new feature (train / valid): 0 0
Training range: 1 to 24
Training rows with a grid penalty: 22 | validation: 9
Correlation with qualifying_position (train): 0.98
 week driver_id  qualifying_position  grid_penalty  estimated_grid_position
    1       D05                    1             5                        6
    1       D12                   17             3                       20
    1       D14                   10             3                       13
    2       D14                    7             3                       10
    2       D19                   17             5                       22


## 11. Controlled comparison: same pipeline, same settings, with and without the feature

Both candidates are built by **one function** with the Section 2 steps and `LogisticRegression(C=1.0, max_iter=1000, random_state=42)`. The only difference is the list of numeric columns. Both are fitted on weeks 1–10 only and evaluated once on the same weeks 11–14. We do **not** run a separate `GridSearchCV` for each version, because choosing a different `C` per version would mix two changes. The dummy is the Week 4 baseline (`most_frequent`).

**Stop 10.** Predict the sign of the difference before running.

In [10]:
def make_pipeline(numeric_cols):
    num_pipe = Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    cat_pipe = Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                         ("encode", OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([("num", num_pipe, numeric_cols), ("cat", cat_pipe, categorical)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(C=1.0, max_iter=1000, random_state=42))])

pipe_without = make_pipeline(numeric).fit(train_f[features_without], train_f.top10)
pipe_with = make_pipeline(numeric_with).fit(train_f[features_with], train_f.top10)
dummy = DummyClassifier(strategy="most_frequent").fit(train_f[features_without], train_f.top10)

# Same model settings in both pipelines (controlled comparison).
assert pipe_without.named_steps["model"].get_params() == pipe_with.named_steps["model"].get_params()
# Same rows: fitted on weeks 1-10 only, scored on weeks 11-14 only.
assert train_f.week.between(1, 10).all() and valid_f.week.between(11, 14).all()
assert len(train_f) == 200 and len(valid_f) == 80

pred_without = pipe_without.predict(valid_f[features_without])
pred_with = pipe_with.predict(valid_f[features_with])
pred_dummy = dummy.predict(valid_f[features_without])

scores = pd.DataFrame({
    "model": ["Week 4 dummy baseline (most_frequent)",
              "Pipeline without feature",
              f"Pipeline with {CANDIDATE}"],
    "numeric inputs": [0, len(numeric), len(numeric_with)],
    "validation balanced accuracy (weeks 11-14)": [
        balanced_accuracy_score(valid_f.top10, pred_dummy),
        balanced_accuracy_score(valid_f.top10, pred_without),
        balanced_accuracy_score(valid_f.top10, pred_with),
    ],
}).round(4)
print(scores.to_string(index=False))
diff = scores.iloc[2, 2] - scores.iloc[1, 2]
print("\nChange from adding the feature:", f"{diff:+.4f}")
print("Validation rows where the two pipelines disagree:", int((pred_without != pred_with).sum()), "of", len(valid_f))
print("Reserved test rows scored in this lab:", 0)
scores

                                model  numeric inputs  validation balanced accuracy (weeks 11-14)
Week 4 dummy baseline (most_frequent)               0                                      0.5000
             Pipeline without feature               4                                      0.8625
Pipeline with estimated_grid_position               5                                      0.8625

Change from adding the feature: +0.0000
Validation rows where the two pipelines disagree: 0 of 80
Reserved test rows scored in this lab: 0


,model,numeric inputs,validation balanced accuracy (weeks 11-14)
0,Week 4 dummy baseline (most_frequent),0,0.5000
1,Pipeline without feature,4,0.8625
2,Pipeline with estimated_grid_position,5,0.8625


### Why no change? A quick look at the coefficients

Because `estimated_grid_position` equals `qualifying_position + grid_penalty`, the model can already express it. The coefficients below show the weight being shared across the three related columns rather than new information appearing.

In [11]:
for name, pipe in [("without", pipe_without), ("with", pipe_with)]:
    names = pipe.named_steps["prep"].get_feature_names_out()
    coefs = pipe.named_steps["model"].coef_[0]
    shown = {n.replace("num__", ""): round(c, 3) for n, c in zip(names, coefs) if n.startswith("num__")}
    print(f"{name:>8}:", shown)

 without: {'qualifying_position': np.float64(-2.168), 'practice_gap_s': np.float64(-1.113), 'previous_top10_rate': np.float64(0.71), 'grid_penalty': np.float64(-0.139)}
    with: {'qualifying_position': np.float64(-1.236), 'practice_gap_s': np.float64(-0.994), 'previous_top10_rate': np.float64(0.787), 'grid_penalty': np.float64(0.082), 'estimated_grid_position': np.float64(-1.196)}


## 12. Decision

**Decision: reject the feature.** On validation weeks 11–14, the pipeline without the feature scored **0.8625** balanced accuracy and the pipeline with `estimated_grid_position` also scored **0.8625**; the Week 4 dummy baseline scored **0.5000**. Not one of the 80 validation predictions changed, so it adds complexity, not value. **Plausible reason:** the new column is an exact sum of two inputs the logistic regression already uses, so it carries no new information; only how the L2 penalty splits the weights changes. **Limitation:** validation has only 80 rows from four weekends and 9 penalised drivers, so one prediction moves the score by about 0.0125; a single split cannot detect a small real effect.

*Word count: 109.*

## 13. AI recommendation audit

We used Claude only to analyse the questions below. We checked each answer before acting on it.

| # | Prompt (summarised) | AI recommendation | Our action | Evidence in notebook |
|---|---|---|---|---|
| 1 | "Is `qualifying_position + grid_penalty` known before the race, or could it leak the result?" | Both columns are known before lights out, so it is valid. Warned that a penalty applied **after** the race would leak; check the column description. | **Accepted.** Section 1 lists `grid_penalty` as pre-race. We added assertions that sources are accepted inputs and never `top10`, `week` or `driver_id`. | Section 10 code cell (`FORBIDDEN`, `assert ... SOURCE_COLS`) |
| 2 | "Will adding this feature help a logistic regression? Should we replace the two source columns with it?" | Expect almost no change, because the feature is an exact linear sum. Suggested **replacing** the two columns with the feature to reduce redundancy. | **Prediction accepted; replacement rejected.** The lab asks for the same pipeline with and without exactly one added feature. Replacing would change three inputs. | Section 9 proposal (expected direction); Section 11 assertion `== {CANDIDATE}` |
| 3 | "How can we be more confident the 0.8625 vs 0.8625 result is not noise?" | (a) Also score weeks 15–18 "to confirm". (b) Count the validation rows where the two pipelines disagree. | **(a) Rejected:** the reserved test must stay sealed in this lab. **(b) Accepted:** 0 of 80 rows differ. | Section 11 output ("disagree", "Reserved test rows scored: 0") |
| 4 | "Review our notebook against this rubric: prediction-time validity 25, controlled comparison 25, reproducible evidence 20, interpretation 20, traceability and AI use 10. What is missing?" | Missing package versions and a runbook; contributor roles not named; check both texts are ≤ 120 words. Also suggested capping the estimated grid at 20. | **Accepted** the first three: added Section 14, named roles in Section 9, added word counts. **Not applied:** the cap. It would change the formula after we saw results; we mention it as an idea only. | Sections 9, 12 (word counts), 14 (versions + runbook) |

**Critical-use note.** Recommendation 3a would have broken the split rule, so we rejected it. AI output is a suggestion; the assertions and the printed table are our evidence.

## 14. Reproducibility: package versions and runbook

**Runbook**
1. Put this notebook and `w04_race_weekends_synthetic_v3.csv` in the same folder.
2. Install: `pip install pandas numpy scikit-learn jupyter` (versions printed below).
3. Open the notebook and choose **Kernel → Restart & Run All**. Or run headless: `python -m nbconvert --to notebook --execute --inplace W04_Thu_Pipelines_Pair_v3.ipynb`.
4. Check Section 11: three rows, scores 0.5000 / 0.8625 / 0.8625, and "Reserved test rows scored in this lab: 0".
5. Every model uses `random_state=42`; there is no other randomness, so scores should match exactly with the same package versions.

In [12]:
import sys, platform, numpy, sklearn
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", numpy.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.14.6
pandas: 3.0.6
numpy: 2.5.3
scikit-learn: 1.9.1
